In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
 
from sklearn.model_selection import GroupKFold, cross_val_score, train_test_split
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb

In [2]:
df = pd.read_csv('Merged_SoilMoisture_Data.csv')

In [3]:
df['Date'] = pd.to_datetime(df['Date'])
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Points: {df['Point_ID'].nunique()}  |  Dates: {df['Date'].nunique()}")

Loaded: 401 rows × 17 columns
Points: 100  |  Dates: 22


In [4]:
df.isnull().sum()

Point_ID                0
Date                    0
latitude                0
longitude               0
Elevation_m             0
Slope                   0
Aspect                  0
Roughness               0
VV_500m                 1
VH_500m                 2
Angle_500m              1
CR=VH/VV                0
RVI                     0
RPI=VV/VH               0
NDPI=(VV-VH)/(VV+VH)    0
MPI                     0
Observed_SM             0
dtype: int64

In [4]:
def engineer_features(df):
    df = df.copy()
 
    #  Temporal features (captures seasonal SM cycle) 
    df['month']     = df['Date'].dt.month
    df['doy']       = df['Date'].dt.dayofyear
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
 
    #  Fix 3 missing SAR values (spatial median per point) 
    for col in ['VV_500m', 'VH_500m', 'Angle_500m']:
        null_mask = df[col].isnull()
        if null_mask.sum() > 0:
            fill = df.groupby('Point_ID')[col].transform('median')
            df.loc[null_mask, col] = fill[null_mask]
 
    #  SAR physics features 
    df['VV_linear']  = 10 ** (df['VV_500m'] / 10)   # dB → linear
    df['VH_linear']  = 10 ** (df['VH_500m'] / 10)
    df['VV_VH_diff'] = df['VV_500m'] - df['VH_500m']
 
    #  Terrain interaction features 
    df['slope_elev']  = df['Elevation_m'] / (df['Slope'] + 0.1)
    df['rough_slope'] = df['Roughness'] * df['Slope']
 
    df['VV_x_season'] = df['VV_linear'] * df['month_sin']
    df['VH_x_season'] = df['VH_linear'] * df['month_sin']
    df['VV_x_cos']    = df['VV_linear'] * df['month_cos']
    df['VH_x_cos']    = df['VH_linear'] * df['month_cos']
 
    return df
 
df = engineer_features(df)
 
FEATURES = [
    # SAR backscatter (raw + linear)
    'VV_500m', 'VH_500m', 'Angle_500m',
    'VV_linear', 'VH_linear', 'VV_VH_diff',
    # Polarimetric indices
    'CR=VH/VV', 'RVI', 'RPI=VV/VH', 'NDPI=(VV-VH)/(VV+VH)', 'MPI',
    # Terrain
    'Elevation_m', 'Slope', 'Aspect', 'Roughness',
    'slope_elev', 'rough_slope',
    # Temporal
    'doy','VV_x_season',
    'VH_x_season',
    'VV_x_cos',   
    'VH_x_cos'    
 
    
]
 
X = df[FEATURES].copy()
y = df['Observed_SM'].copy()

In [5]:
unique_points = df['Point_ID'].unique()
np.random.seed(42)
test_points = np.random.choice(unique_points,
                                size=int(len(unique_points) * 0.2),
                                replace=False)
 
train_mask = ~df['Point_ID'].isin(test_points)
test_mask  =  df['Point_ID'].isin(test_points)
 
X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]
 
print(f"\nTrain: {X_train.shape[0]} rows  |  Test: {X_test.shape[0]} rows")
print(f"Test spatial points held out: {sorted(test_points)}")


Train: 323 rows  |  Test: 78 rows
Test spatial points held out: [np.int64(1), np.int64(7), np.int64(8), np.int64(11), np.int64(18), np.int64(27), np.int64(32), np.int64(38), np.int64(42), np.int64(46), np.int64(48), np.int64(53), np.int64(58), np.int64(79), np.int64(80), np.int64(83), np.int64(86), np.int64(92), np.int64(96), np.int64(100)]


In [6]:
scaler     = RobustScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

In [10]:
from sklearn.svm import SVR
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import loguniform
import numpy as np


# TUNE SVR

gkf          = GroupKFold(n_splits=5)
groups_train = df.loc[train_mask, 'Point_ID'].values

param_dist = {
    'C':       loguniform(0.1, 1000),
    'gamma':   loguniform(1e-4, 10),
    'epsilon': loguniform(0.001, 0.5),
}

svr_search = RandomizedSearchCV(
    SVR(kernel='rbf'),
    param_distributions=param_dist,
    n_iter=100,
    cv=gkf,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)
svr_search.fit(X_train_sc, y_train, groups=groups_train)

print(f"Best params : {svr_search.best_params_}")
print(f"Best CV R²  : {svr_search.best_score_:.4f}")


# EVALUATE

best_svr   = svr_search.best_estimator_
train_pred = best_svr.predict(X_train_sc)
test_pred  = best_svr.predict(X_test_sc)

train_m = full_metrics(y_train, train_pred, "TRAIN")
test_m  = full_metrics(y_test,  test_pred,  "TEST (spatial holdout)")


# 5-FOLD SPATIAL CV

print(f"\n{'─'*35}")
print(f"  5-FOLD SPATIAL CV")
print(f"{'─'*35}")

fold_r2, fold_rmse, fold_mae = [], [], []

for fold, (tr_idx, val_idx) in enumerate(
        gkf.split(X, y, groups=df['Point_ID']), 1):

    X_tr  = X.iloc[tr_idx].values
    X_val = X.iloc[val_idx].values
    y_tr  = y.iloc[tr_idx].values
    y_val = y.iloc[val_idx].values

    sc       = RobustScaler()
    X_tr_sc  = sc.fit_transform(X_tr)
    X_val_sc = sc.transform(X_val)

    svr = SVR(kernel='rbf', **svr_search.best_params_)
    svr.fit(X_tr_sc, y_tr)
    val_pred = svr.predict(X_val_sc)

    f_r2   = r2_score(y_val, val_pred)
    f_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    f_mae  = mean_absolute_error(y_val, val_pred)
    fold_r2.append(f_r2)
    fold_rmse.append(f_rmse)
    fold_mae.append(f_mae)

    print(f"  Fold {fold}  →  R²={f_r2:.4f}  RMSE={f_rmse:.4f}  MAE={f_mae:.4f}")

print(f"{'─'*35}")
print(f"  Mean  →  R²={np.mean(fold_r2):.4f}  RMSE={np.mean(fold_rmse):.4f}  MAE={np.mean(fold_mae):.4f}")
print(f"  Std   →  R²={np.std(fold_r2):.4f}  RMSE={np.std(fold_rmse):.4f}  MAE={np.std(fold_mae):.4f}")

Best params : {'C': np.float64(12.329098365270504), 'epsilon': np.float64(0.014253462720121492), 'gamma': np.float64(0.0001339971723117974)}
Best CV R²  : 0.1047

───────────────────────────────────
  TRAIN
───────────────────────────────────
  R²     : 0.2997
  R      : 0.5610
  RMSE   : 0.0677 m³/m³
  ubRMSE : 0.0670 m³/m³
  MAE    : 0.0522 m³/m³
  Bias   : -0.0099 m³/m³

───────────────────────────────────
  TEST (spatial holdout)
───────────────────────────────────
  R²     : 0.1105
  R      : 0.4118
  RMSE   : 0.0802 m³/m³
  ubRMSE : 0.0784 m³/m³
  MAE    : 0.0624 m³/m³
  Bias   : -0.0168 m³/m³

───────────────────────────────────
  5-FOLD SPATIAL CV
───────────────────────────────────
  Fold 1  →  R²=0.0899  RMSE=0.0690  MAE=0.0507
  Fold 2  →  R²=0.3399  RMSE=0.0595  MAE=0.0491
  Fold 3  →  R²=-0.1665  RMSE=0.0859  MAE=0.0703
  Fold 4  →  R²=0.2211  RMSE=0.0849  MAE=0.0687
  Fold 5  →  R²=0.0396  RMSE=0.0823  MAE=0.0631
───────────────────────────────────
  Mean  →  R²=0.1048  R